In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Defining the state

class BatsmanState(TypedDict):
    runs: int
    balls_faced: int
    fours: int
    sixs: int 
    
    strike_rate: float
    balls_per_boundary: float
    boundary_percentage: float
    summary: str

In [ ]:
def calculate_sr(state: BatsmanState):
    strike_rate = (state['runs'] / state['balls_faced']) * 100
    
    return {"strike_rate": strike_rate}

In [ ]:
def calculate_balls_per_boundary(state: BatsmanState):
    balls_per_boundary = ((state['fours'] + state['sixs']) / state['balls_faced'])
    
    return {"balls_per_boundary": balls_per_boundary}


In [ ]:
def calculate_boundary_percentage(state: BatsmanState):
    boundary_percentage = (((state['fours'] * 4) + (state['sixs'] * 6)) / state['runs']) * 100
    
    return {"boundary_percentage": boundary_percentage}

In [ ]:
def generate_summary(state: BatsmanState):
    summary = f"""
        Strike rate -> {state['strike_rate']}
        Boundary Percentage -> {state['boundary_percentage']}
        Balls Per boundary -> {state['balls_per_boundary']}

    """
    
    return {"summary": summary}

In [ ]:
# Adding nodes & edges to the graph
# Defining the graph

graph = StateGraph(BatsmanState)

graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_balls_per_boundary", calculate_balls_per_boundary)
graph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
graph.add_node("generate_summary", generate_summary)

# Adding edges to the graph

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_balls_per_boundary')
graph.add_edge(START, 'calculate_boundary_percentage')

graph.add_edge('calculate_sr', 'generate_summary')
graph.add_edge('calculate_balls_per_boundary', 'generate_summary')
graph.add_edge('calculate_boundary_percentage', 'generate_summary')

graph.add_edge('generate_summary', END)

workflow = graph.compile()
workflow




In [ ]:
initial_state = {
    "runs": 100,
    "balls_faced": 50,
    "sixs": 6,
    "fours": 4
}

final_result = workflow.invoke(initial_state)
print(final_result)